# Laboratorio — Sistemas de Recomendación (15 min)
**Curso:** Aprendizaje Automático · MCDA 2026-1
**Sesión:** Sistemas de recomendación: content-based, collaborative filtering y factorización de matrices

## Objetivo
Construir un recomendador desde cero usando los **tres enfoques clásicos**: filtrado basado en contenido, filtrado colaborativo por vecinos y factorización de matrices (SVD). Comparar sus recomendaciones y evaluarlas con métricas de **rating** (RMSE) y de **ranking** (Precision@K).

## Dataset
**MovieLens 100K** — versión reducida y reproducible que generamos a partir de un subconjunto sintético inspirado en el dataset real: 50 usuarios, 30 películas con géneros, ~600 ratings entre 1 y 5. Lo construimos en la primera celda para que el lab corra sin descargas externas.

## Agenda (15 min)
| Paso | Tiempo | Qué haces |
|------|--------|-----------|
| 1. Generar datos y explorar la matriz de utilidad | 2 min | Ver sparsity |
| 2. Content-based filtering | 3 min | Similitud coseno sobre géneros |
| 3. Collaborative filtering (item-based KNN) | 3 min | Vecinos sobre ratings |
| 4. Factorización de matrices (SVD) | 3 min | Espacio latente + RMSE |
| 5. Retos individuales | 4 min | 3 micro-retos |

## Paso 1 · Generar datos y explorar la matriz de utilidad (2 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# --- Catálogo de películas con géneros (vector binario) ---
generos = ['accion', 'drama', 'comedia', 'sci_fi', 'romance', 'thriller']
n_items = 30
n_users = 50

# Cada película tiene 1-3 géneros activos
items_matrix = np.zeros((n_items, len(generos)), dtype=int)
for i in range(n_items):
    k = rng.integers(1, 4)
    items_matrix[i, rng.choice(len(generos), size=k, replace=False)] = 1

items = pd.DataFrame(items_matrix, columns=generos,
                     index=[f'movie_{i:02d}' for i in range(n_items)])

# --- Cada usuario tiene un perfil de gustos latente sobre los géneros ---
user_taste = rng.uniform(0, 1, size=(n_users, len(generos)))

# --- Generar ratings: cada usuario califica ~12 películas elegidas con sesgo a sus gustos ---
ratings_list = []
for u in range(n_users):
    afinidad = items_matrix @ user_taste[u]            # afinidad usuario-película
    prob = afinidad / afinidad.sum()
    n_rated = rng.integers(8, 16)
    vistas = rng.choice(n_items, size=n_rated, replace=False, p=prob)
    for i in vistas:
        score = afinidad[i] / afinidad.max() * 4 + 1   # mapear a 1-5
        score = np.clip(score + rng.normal(0, 0.5), 1, 5)
        ratings_list.append((f'user_{u:02d}', f'movie_{i:02d}', round(score * 2) / 2))

ratings = pd.DataFrame(ratings_list, columns=['user', 'movie', 'rating'])

# Matriz de utilidad (usuarios x películas)
R = ratings.pivot(index='user', columns='movie', values='rating')

print(f'Usuarios: {R.shape[0]} · Películas: {R.shape[1]} · Ratings: {len(ratings)}')
print(f'Densidad de la matriz: {R.notna().sum().sum() / R.size * 100:.1f} %')
print('\nPrimeras filas de la matriz de utilidad (NaN = sin rating):')
R.iloc[:6, :8]

In [ ]:
# Visualizar la sparsity
plt.figure(figsize=(10, 4))
plt.imshow(R.notna().values, aspect='auto', cmap='Blues')
plt.xlabel('Películas'); plt.ylabel('Usuarios')
plt.title(f'Mapa de la matriz de utilidad — azul = rating observado · blanco = NaN')
plt.tight_layout(); plt.show()

**Pregunta rápida:** ¿qué porcentaje de la matriz está vacío? Recordá que en producción real este valor suele ser **> 99 %**. Predecir esos huecos *es* el problema de recomendación.

## Paso 2 · Content-based filtering (3 min)
**Idea:** representar cada película como un vector de géneros y construir el perfil de cada usuario como el **promedio ponderado por rating** de las películas que ha visto. Recomendar lo más similar (coseno) al perfil.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def perfil_usuario(user_id, R, items):
    """Promedio ponderado por rating de los vectores de género de las películas vistas."""
    vistas = R.loc[user_id].dropna()
    if len(vistas) == 0:
        return np.zeros(items.shape[1])
    pesos = vistas.values
    return (items.loc[vistas.index].values.T @ pesos) / pesos.sum()

def recomendar_content(user_id, R, items, top_k=5):
    perfil = perfil_usuario(user_id, R, items).reshape(1, -1)
    sims = cosine_similarity(perfil, items.values).ravel()
    scores = pd.Series(sims, index=items.index)
    # Excluir lo ya visto
    vistas = R.loc[user_id].dropna().index
    return scores.drop(vistas).sort_values(ascending=False).head(top_k)

user_demo = 'user_00'
print(f'Películas vistas por {user_demo} (rating, géneros):')
for m, r in R.loc[user_demo].dropna().sort_values(ascending=False).head(5).items():
    g = items.loc[m][items.loc[m] == 1].index.tolist()
    print(f'  {m}  rating={r}  géneros={g}')

print(f'\nTop-5 recomendaciones content-based para {user_demo}:')
recs_content = recomendar_content(user_demo, R, items, top_k=5)
for m, s in recs_content.items():
    g = items.loc[m][items.loc[m] == 1].index.tolist()
    print(f'  {m}  sim={s:.3f}  géneros={g}')

**Observa:** las películas recomendadas comparten muchos géneros con las mejor calificadas por el usuario. Es **explicable** ("te recomiendo X porque te gustó Y, ambas son sci-fi") pero corre riesgo de *filter bubble*: nunca propondrá un género que el usuario no haya tocado.

## Paso 3 · Collaborative filtering — item-based KNN (3 min)
**Idea:** dos películas se parecen si los **mismos usuarios** las calificaron parecido. Para predecir el rating de un usuario sobre una película no vista, miramos las películas similares que **sí** calificó.

In [ ]:
# Matriz item x user con NaN -> 0 para calcular similitudes
R_filled = R.fillna(0).values             # users x items
item_sim_arr = cosine_similarity(R_filled.T)  # items x items
np.fill_diagonal(item_sim_arr, 0)         # un ítem no es vecino de sí mismo
item_sim = pd.DataFrame(item_sim_arr, index=R.columns, columns=R.columns)

def predecir_rating_cf(user_id, movie_id, R, item_sim, k=5):
    """Predice el rating como promedio ponderado por similitud de los k vecinos vistos."""
    vistas = R.loc[user_id].dropna()
    sims = item_sim.loc[movie_id, vistas.index]
    top = sims.nlargest(k)
    if top.sum() == 0:
        return vistas.mean()             # fallback: media del usuario
    return (top * vistas[top.index]).sum() / top.sum()

def recomendar_cf(user_id, R, item_sim, top_k=5, k_vecinos=5):
    no_vistas = R.loc[user_id][R.loc[user_id].isna()].index
    preds = {m: predecir_rating_cf(user_id, m, R, item_sim, k=k_vecinos) for m in no_vistas}
    return pd.Series(preds).sort_values(ascending=False).head(top_k)

print(f'Top-5 recomendaciones collaborative (item-based KNN) para {user_demo}:')
recs_cf = recomendar_cf(user_demo, R, item_sim, top_k=5, k_vecinos=5)
for m, s in recs_cf.items():
    g = items.loc[m][items.loc[m] == 1].index.tolist()
    print(f'  {m}  pred_rating={s:.2f}  géneros={g}')

**Observa:** este enfoque **no usa los géneros**, solo patrones de co-rating. Puede sorprenderte recomendando algo de un género no esperado si otros usuarios con gustos parecidos lo disfrutaron — eso es *serendipia*.

## Paso 4 · Factorización de matrices con SVD (3 min)
**Idea:** aproximar `R ≈ U · Vᵀ` con `k` factores latentes. Cada usuario y cada ítem viven en el mismo espacio de `k` dimensiones; el rating predicho es el producto interno. Evaluamos con **RMSE** sobre un test de ratings observados.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Train/test sobre los pares (user, movie, rating) observados
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Reconstruir la matriz de train, rellenando NaN con la media global (para que SVD funcione)
R_train = train.pivot(index='user', columns='movie', values='rating')
R_train = R_train.reindex(index=R.index, columns=R.columns)   # mismas dimensiones
media_global = train['rating'].mean()
R_train_filled = R_train.fillna(media_global).values

# Centrar antes de descomponer
R_centered = R_train_filled - media_global

# SVD truncada con k factores
k = 8
U, sigma, Vt = np.linalg.svd(R_centered, full_matrices=False)
U_k     = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k    = Vt[:k, :]

# Reconstrucción aproximada
R_hat = U_k @ sigma_k @ Vt_k + media_global
R_hat = pd.DataFrame(R_hat, index=R.index, columns=R.columns)

# Evaluar RMSE en el test
y_true = test['rating'].values
y_pred = np.array([R_hat.loc[u, m] for u, m in zip(test['user'], test['movie'])])
rmse_svd = np.sqrt(mean_squared_error(y_true, y_pred))

# Baseline: predecir siempre la media global
rmse_baseline = np.sqrt(mean_squared_error(y_true, np.full_like(y_true, media_global)))

print(f'RMSE baseline (media global): {rmse_baseline:.3f}')
print(f'RMSE SVD (k={k} factores)    : {rmse_svd:.3f}')
print(f'Mejora: {(rmse_baseline - rmse_svd) / rmse_baseline * 100:.1f} %')

In [ ]:
# Recomendaciones del modelo SVD para el mismo usuario
def recomendar_svd(user_id, R, R_hat, top_k=5):
    no_vistas = R.loc[user_id][R.loc[user_id].isna()].index
    return R_hat.loc[user_id, no_vistas].sort_values(ascending=False).head(top_k)

print(f'Top-5 recomendaciones SVD para {user_demo}:')
recs_svd = recomendar_svd(user_demo, R, R_hat, top_k=5)
for m, s in recs_svd.items():
    g = items.loc[m][items.loc[m] == 1].index.tolist()
    print(f'  {m}  pred_rating={s:.2f}  géneros={g}')

# Comparar las 3 listas
comparacion = pd.DataFrame({
    'content_based': recs_content.index.tolist(),
    'collaborative': recs_cf.index.tolist(),
    'svd'          : recs_svd.index.tolist(),
})
print('\nComparación de Top-5 entre los tres enfoques:')
comparacion

**Observa:** los tres métodos producen listas distintas. No hay un "ganador" universal — en producción, sistemas reales **combinan** content-based (para cold start de ítems), collaborative (para serendipia) y modelos latentes (para precisión).

## Paso 5 · Retos individuales (4 min)

### Reto 1 · Precision@5 con relevancia binaria
Vamos a evaluar el SVD como ranker. El `R_hat` ya está entrenado **solo con `train`**, así que las interacciones de `test` son ítems "no vistos" desde la perspectiva del modelo. Define como **relevante** un rating real ≥ 4 en `test`. Para cada usuario con al menos 1 ítem relevante en test:
1. Ordena **todos los ítems que el usuario no calificó en train** por `R_hat`.
2. Toma el Top-5.
3. Calcula `hits / 5` donde `hits` es la intersección con sus relevantes en test.

Reporta la **Precision@5 promedio**.

*Pista:* `no_vistas_train = R_train.loc[u][R_train.loc[u].isna()].index`

### Reto 2 · Variar el número de factores latentes `k`
Repite el entrenamiento de SVD con `k ∈ {2, 4, 8, 16, 24}` y grafica el RMSE en test. ¿Hay un valor de `k` óptimo? ¿Qué pasa cuando `k` es demasiado grande (overfitting al ruido)?

### Reto 3 · Cold start de ítem
¿Cuál de los tres enfoques (content-based, collaborative item-based, SVD) **podría** recomendar una película que nadie ha calificado y cuáles **no**? Justifica en una línea. Para validar, simula que `movie_29` es nueva: borra todos sus ratings y verifica si aparece en el Top-10 de cada método para `user_05`.

*Pista:* `R_cold = R.copy(); R_cold['movie_29'] = np.nan`

In [ ]:
# Reto 1 — tu código aquí
# pista:
# relevantes_por_usuario = test[test['rating'] >= 4].groupby('user')['movie'].apply(set)
# para cada user u en relevantes_por_usuario:
#     no_vistas_train = R_train.loc[u][R_train.loc[u].isna()].index
#     top5 = R_hat.loc[u, no_vistas_train].sort_values(ascending=False).head(5).index
#     hits = len(set(top5) & relevantes_por_usuario[u])
#     precision = hits / 5


In [ ]:
# Reto 2 — tu código aquí


In [ ]:
# Reto 3 — tu respuesta y código aquí
# pista:
# R_cold = R.copy()
# R_cold['movie_29'] = np.nan        # nadie la ha calificado
# luego reconstruye item_sim y R_hat usando R_cold y mira si movie_29 aparece
# en cada uno de los tres rankings para 'user_05'


## Cierre
Construiste un recomendador con los **tres enfoques fundamentales**: content-based usa atributos del ítem y funciona desde el día 1 pero genera *filter bubble*; collaborative filtering aprovecha la *inteligencia colectiva* y produce serendipia pero sufre con cold start; la factorización de matrices proyecta usuarios e ítems al mismo **espacio latente** y es la base de modelos modernos (ALS, two-tower, DLRM). En la práctica, los sistemas industriales combinan los tres en una arquitectura por etapas: **retrieval → ranking → re-ranking**.